#### Week 5 Day 4
AutoGen Core - Distributed

I'm only going to give a Teaser of this!!

Partly because I'm unsure how relevant it is to you. If you'd like me to add more content for this, please do let me know..

In [1]:
from dataclasses import dataclass
from autogen_core import AgentId, MessageContext, RoutedAgent, message_handler
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.messages import TextMessage
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_ext.tools.langchain import LangChainToolAdapter
from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain.agents import Tool
from IPython.display import display, Markdown

from dotenv import load_dotenv

load_dotenv(override=True)

ALL_IN_ONE_WORKER = False

### Start with our Message class

In [2]:
@dataclass
class Message:
    content: str

#### And now - a host for our distributed runtime

In [3]:
from autogen_ext.runtimes.grpc import GrpcWorkerAgentRuntimeHost

host = GrpcWorkerAgentRuntimeHost(address="localhost:50051")
host.start() 

#### Let's reintroduce a tool

In [4]:
serper = GoogleSerperAPIWrapper()
langchain_serper =Tool(name="internet_search", func=serper.run, description="Useful for when you need to search the internet")
autogen_serper = LangChainToolAdapter(langchain_serper)

In [5]:
instruction1 = "To help with a decision on whether to use AutoGen in a new AI Agent project, \
please research and briefly respond with reasons in favor of choosing AutoGen; the pros of AutoGen."

instruction2 = "To help with a decision on whether to use AutoGen in a new AI Agent project, \
please research and briefly respond with reasons against choosing AutoGen; the cons of Autogen."

judge = "You must make a decision on whether to use AutoGen for a project. \
Your research team has come up with the following reasons for and against. \
Based purely on the research from your team, please respond with your decision and brief rationale."

#### And make some Agents

In [6]:
class Player1Agent(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        self._delegate = AssistantAgent(name, model_client=model_client, tools=[autogen_serper], reflect_on_tool_use=True)

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], ctx.cancellation_token)
        return Message(content=response.chat_message.content)
    
class Player2Agent(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        self._delegate = AssistantAgent(name, model_client=model_client, tools=[autogen_serper], reflect_on_tool_use=True)

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], ctx.cancellation_token)
        return Message(content=response.chat_message.content)
    
class Judge(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        self._delegate = AssistantAgent(name, model_client=model_client)
        
    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        message1 = Message(content=instruction1)
        message2 = Message(content=instruction2)
        inner_1 = AgentId("player1", "default")
        inner_2 = AgentId("player2", "default")
        response1 = await self.send_message(message1, inner_1)
        response2 = await self.send_message(message2, inner_2)
        result = f"## Pros of AutoGen:\n{response1.content}\n\n## Cons of AutoGen:\n{response2.content}\n\n"
        judgement = f"{judge}\n{result}Respond with your decision and brief explanation"
        message = TextMessage(content=judgement, source="user")
        response = await self._delegate.on_messages([message], ctx.cancellation_token)
        return Message(content=result + "\n\n## Decision:\n\n" + response.chat_message.content)

In [7]:
from autogen_ext.runtimes.grpc import GrpcWorkerAgentRuntime

if ALL_IN_ONE_WORKER:

    worker = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker.start()

    await Player1Agent.register(worker, "player1", lambda: Player1Agent("player1"))
    await Player2Agent.register(worker, "player2", lambda: Player2Agent("player2"))
    await Judge.register(worker, "judge", lambda: Judge("judge"))

    agent_id = AgentId("judge", "default")

else:

    worker1 = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker1.start()
    await Player1Agent.register(worker1, "player1", lambda: Player1Agent("player1"))

    worker2 = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker2.start()
    await Player2Agent.register(worker2, "player2", lambda: Player2Agent("player2"))

    worker = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker.start()
    await Judge.register(worker, "judge", lambda: Judge("judge"))
    agent_id = AgentId("judge", "default")

In [8]:
response = await worker.send_message(Message(content="Go!"), agent_id)

In [9]:
display(Markdown(response.content))

## Pros of AutoGen:
Here are some pros of using AutoGen for your AI agent project:

1. **Multi-Agent Orchestration**: AutoGen excels at creating and managing multi-agent systems, allowing for improved collaboration among different agents to solve complex problems.

2. **Efficiency and Time-Saving**: Users have reported significant reductions in time spent on tasks, such as a 75% reduction in bid writing time, which can be crucial for projects with tight deadlines.

3. **Customizability**: The platform provides flexibility for developers to customize agents for specific tasks, ensuring that the system is tailored to meet unique project requirements.

4. **Natural Language Coordination**: AutoGen simplifies inter-agent communication by using natural language, which reduces the complexity involved in coordinating different agents and eliminates the need for custom protocols.

5. **Enhanced Problem-Solving**: The conversational capabilities of AutoGen facilitate divergent thinking, improving the agents' ability to reason and provide accurate responses.

6. **Support for Various Operational Models**: AutoGen can support both fully autonomous operations and hybrid models where human oversight is integrated, making it suitable for diverse application contexts.

These benefits highlight why AutoGen may be a strong choice for enhancing the efficiency and effectiveness of AI agent projects. 

TERMINATE

## Cons of AutoGen:
Here are some cons of using AutoGen for your AI Agent project:

1. **Limited AI Capabilities**: AutoGen may not be suitable for more complex AI-driven automation tasks, as it can struggle with intricate problem-solving that requires advanced algorithms.

2. **Less Customizable**: The platform offers limited flexibility in customization, which may hinder the ability to tailor solutions to specific project needs.

3. **Cost Issues**: Usage-based pricing can escalate quickly, especially if workflows are not optimized for efficiency. There is often no pay-as-you-go option, leading to the necessity of upgrading even if the current usage does not justify it.

4. **Dependency on Collaborative Agents**: AutoGen is primarily designed for scenarios where multiple AI agents work dynamically together, which might not align with all project requirements.

5. **Limited Community Support**: Compared to more established frameworks, AutoGen might have less community support and documentation, making it harder to find help or resources during development.

These drawbacks should be carefully considered in the context of the project's specific goals and requirements. 

TERMINATE



## Decision:

After considering the pros and cons of using AutoGen for the project, I recommend moving forward with AutoGen. 

The primary rationale for this decision lies in its strengths in multi-agent orchestration and efficiency. The significant time savings reported by users, along with the ability to customize agents for specific tasks, align well with our project's needs for improved collaboration and rapid execution. Additionally, the natural language coordination capability simplifies interactions between agents, which is a valuable feature for enhancing communication in complex scenarios.

While there are some limitations, such as potentially high costs and limited AI capabilities for very complex tasks, the benefits of improved problem-solving and operational flexibility make AutoGen a compelling option for our current objectives.

TERMINATE

In [10]:
await worker.stop()
if not ALL_IN_ONE_WORKER:
    await worker1.stop()
    await worker2.stop()

In [11]:
await host.stop()